# OCR Error and How It Propagates

**Computational Text Analysis Pack · Module 14 — Preparing Text, Properly**

*Technique:* Character error rate; measuring the downstream cost of noisy text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/14d_ocr_error_propagation.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Every corpus in this course arrived as clean text. Real archival corpora do not:
they arrive as images, pass through OCR or HTR, and carry an error rate that
propagates through tokenisation, counting, tagging and every result you report.

The survey is blunt about it — "OCR/HTR error propagates through every
downstream step" — and sets a working threshold: **noise below a few percent
CER is roughly where frequency work stays reliable.**

We cannot run an OCR engine in a browser sandbox. We can do the thing that
actually teaches the lesson: take clean text, damage it the way a scanner
damages text, and measure what breaks.

## The tooling

For print, **Tesseract** (open source) and **ABBYY FineReader** (commercial).
For manuscripts and connected scripts, HTR has largely replaced template OCR:
**Transkribus** (server-based, upload required, PyLaia models trained inside it
currently cannot be exported) and **eScriptorium** (open source, built on the
**Kraken** engine). Kraken remains the best open engine for connected scripts,
which matters for Arabic-script and Ajami materials in African archives.

None of them run here. All of them produce output whose quality you must report,
and CER is how you report it.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import random

# Confusion pairs a real scanner produces: shapes that look alike at low DPI.
CONFUSIONS = [("rn", "m"), ("m", "rn"), ("l", "1"), ("1", "l"), ("O", "0"),
              ("o", "c"), ("c", "e"), ("e", "c"), ("i", "j"), ("t", "f"),
              ("h", "b"), ("s", "8"), ("u", "ii"), ("d", "cl")]

def degrade(text, rate, seed=0):
    """Apply substitutions at roughly `rate` per character."""
    rng = random.Random(seed)
    out, i = [], 0
    while i < len(text):
        if rng.random() < rate:
            hit = [(a, b) for a, b in CONFUSIONS if text.startswith(a, i)]
            if hit:
                a, b = rng.choice(hit)
                out.append(b)
                i += len(a)
                continue
            if rng.random() < 0.3:      # a dropped character
                i += 1
                continue
        out.append(text[i])
        i += 1
    return "".join(out)

clean = " ".join(df_colonial["text"].head(300))
print(f"clean sample: {len(clean):,} characters")
print()
print("clean   :", clean[:150])
print("degraded:", degrade(clean[:150], 0.06, seed=1))

## Character error rate

CER is edit distance between the OCR output and the ground truth, divided by the
length of the truth. WER is the same at word level. Both are what an HTR project
reports; neither tells you what the errors *cost*, which is the next section.

In [ ]:
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

truth = clean[:4000]
print(f"{'noise rate':>12}{'CER':>10}{'WER':>10}")
print("-" * 32)
levels = [0.005, 0.01, 0.02, 0.05, 0.10]
measured = []
for rate in levels:
    noisy = degrade(truth, rate, seed=2)
    cer = edit_distance(truth, noisy) / len(truth)
    tw, nw = truth.split(), noisy.split()
    wer = edit_distance(tw, nw) / len(tw)
    measured.append((rate, cer, wer))
    print(f"{rate:>12.3f}{cer:>10.3f}{wer:>10.3f}")
print()
print("WER runs several times higher than CER: one bad character ruins a whole")
print("word. Frequency analysis works on words, so WER is the number that bites.")

## What the noise costs downstream

Now the part that matters for a humanities argument. Build a frequency list from
clean text and from damaged text, and measure how much of the top of the list
survives.

In [ ]:
base = Counter(tokenise(clean))
top_clean = [w for w, _ in base.most_common(100)]

rows = []
for rate in levels:
    noisy = Counter(tokenise(degrade(clean, rate, seed=3)))
    top_noisy = [w for w, _ in noisy.most_common(100)]
    overlap = len(set(top_clean) & set(top_noisy))
    ghosts = sum(1 for w in noisy if w not in base)
    rows.append({"noise": rate, "top100_kept": overlap,
                 "vocab_clean": len(base), "vocab_noisy": len(noisy),
                 "invented_words": ghosts})
dmg = pd.DataFrame(rows)
print(dmg.to_string(index=False))
print()
print("Invented words are the real damage: every OCR error creates a new type")
print("that never existed. Vocabulary inflates, richness measures break, and a")
print("keyness list fills with words no one wrote.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([m[0] for m in measured], [m[1] for m in measured], "-o", label="CER")
axes[0].plot([m[0] for m in measured], [m[2] for m in measured], "-s", label="WER")
axes[0].axhline(0.03, ls="--", color="grey", lw=1)
axes[0].set_xlabel("simulated noise rate")
axes[0].set_ylabel("error rate")
axes[0].set_title("CER vs WER (dashed: ~3% reliability threshold)")
axes[0].legend()
axes[1].plot(dmg["noise"], dmg["invented_words"], "-o", color="crimson")
axes[1].set_xlabel("simulated noise rate")
axes[1].set_ylabel("word types that never existed")
axes[1].set_title("OCR invents vocabulary")
plt.tight_layout()

### 🔧 Your turn 1

At which noise level does the top-100 frequency list lose more than ten words? What would you have to report about a corpus at that CER?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

On most runs the top-100 list holds up surprisingly well until around 5% noise,
because the most frequent words are short function words with few confusable
letters. That is a trap rather than a reassurance: the headline frequency list
looks healthy while the mid- and low-frequency ranges — exactly where keyness,
Zeta and topic models do their work — are already badly corrupted. A corpus at
that CER can support a claim about *the*, and cannot support a claim about a
content word without a manual check of the concordance lines behind it.

</details>